# Речной сток для природы и орошения

*Правило экологического стока в пяти блоках TaqSim с оценкой через fishy*

Д-р Тобиас Зигфрид, hydrosolutions GmbH · CC BY 4.0

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

# Вопрос

Река питает канал, а канал — поля Самарканда. Всё, что канал не забирает, остаётся в реке и течёт дальше. Оператор гидроузла решает не только, сколько воды оставить, но и как это сделать:

- **Оставлять долю.** Каждый день фиксированная доля суточного стока остаётся в реке, независимо от его величины.
- **Гарантировать минимум.** Каждый день первые заданные кубометры в секунду остаются в реке; канал забирает только превышение.

Оба правила могут дать культурам одинаковый годовой объём, но создают разный режим реки. Этот блокнот строит минимальную модель TaqSim, показывающую разницу на реальном Зарафшане у Раватходжи, и оценивает реку индексом изменения гидрологического режима.

**Что вы узнаете**

- Как работает правило экологического стока TaqSim (`EFlowSplit`) и как связать его с источником, каналом и конечным узлом.
- Как индекс изменения режима (IARI из fishy, основанный на 33 показателях IHA) превращает суточный ряд в число и чего это число не отражает.
- Почему при одинаковом водоснабжении культур гарантированный минимум сохраняет более близкий к естественному режим, чем доля: важна форма правила, а не только объём.

In [ ]:
# @title Подготовка: сначала выполните эту ячейку
# ── Подготовка: сначала выполните эту ячейку. In Colab it takes about a minute. ──────────────────────────────────
# Загружает папку курса, устанавливает TaqSim и сообщает Python путь к коду курса.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # ещё не в папке курса (обычно так в Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
# В ранее использованной среде может быть старая копия курса: обновляем папку и
# удаляем уже загруженные модули курса, чтобы импортировать текущий код.
if os.path.exists(".git"):
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
for name in [m for m in sys.modules if m == "zarafshan_taqsim" or m.startswith("zarafshan_taqsim.")]:
    del sys.modules[name]
try:
    import fishy  # noqa: F401
except ImportError:
    %pip install -q polars "git+https://github.com/hydrosolutions/fishy.git@881660e8901987d63d89caf61ef0d30991000301"
sys.path[:0] = [os.path.abspath("eflow_game/src")]
print("Готово. Рабочая папка:", os.getcwd())


In [ ]:
# @title Стандартные инструменты
#| label: setup
# Стандартные инструменты
import inspect  # для показа исходного кода функции в блокноте
import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl  # таблицы; новые TaqSim и fishy работают с polars
from IPython.display import Markdown

# Эталонная модель игры (workshop/eflow_game/src/eflow_game/model.py): система TaqSim, оценки fishy,
# сетка планов и экспорт. Все результаты игры вычислены этими функциями.
from eflow_game import model as m
from eflow_game.model import Plan, build_system, demand_curve, load_inflow, run_year, score_river, twin_year

logging.basicConfig(level=logging.WARNING)
DAY = 86400.0

# Готовая сетка: каждый план рассчитан в TaqSim за 2019–2023 и оценён fishy
DATA = json.loads(m.DATA_FILE.read_text())
PLANS = pl.DataFrame(DATA["plans"]).select(
    "id", "name", "share_pct", "ceiling_m3s", "crops_pct", "score", "score_is_total", "scored", "classification", "unbeaten", "lowest_flow_m3s"
)
META = DATA["meta"]
Markdown(
    f"Загружено планов: {len(DATA['plans'])} ({META['runs']} годовых расчётов TaqSim по {META['mean_run_seconds']:.1f} с, "
    f"процессов: {META['workers']}, всего {META['wall_seconds_total'] / 60:.1f} мин; создано {META['built']})."
)

In [ ]:
# @title Рисунок: модель как сеть
#| label: fig-network
#| fig-cap: "Модель в виде сети. Вода течёт слева направо. Правила задаются двум участкам; конечные узлы лишь записывают поступления."
#| echo: false
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.set_xlim(0, 10); ax.set_ylim(0, 3.2); ax.axis("off")
boxes = {
    "inflow": (0.9, 1.6, "Приток\nSOURCE", "#4a6d8c"),
    "headworks": (3.3, 1.6, "Гидроузел\nREACH · EFlowSplit", "#0b0b0b"),
    "canal_gate": (5.8, 1.6, "Водозабор\nREACH · PriorityDistribution", "#0b0b0b"),
    "fields": (8.6, 2.5, "Поля\nSINK", "#007a00"),
    "river_end": (8.6, 0.7, "Конец реки\nSINK", "#0f7f58"),
}
for key, (x, y, label, colour) in boxes.items():
    ax.add_patch(plt.Rectangle((x - 0.85, y - 0.42), 1.7 if key not in ("headworks", "canal_gate") else 2.0, 0.84, fc="white", ec=colour, lw=1.6))
    ax.text(x + (0 if key not in ("headworks", "canal_gate") else 0.15), y, label, ha="center", va="center", fontsize=9)
arrow = dict(arrowstyle="-|>", lw=1.8)
ax.annotate("", (2.45, 1.6), (1.75, 1.6), arrowprops=dict(color="#4a6d8c", **arrow))
ax.annotate("", (4.95, 1.6), (4.45, 1.6), arrowprops=dict(color="#4a6d8c", **arrow))
ax.annotate("", (7.75, 2.5), (7.0, 1.75), arrowprops=dict(color="#007a00", **arrow))
ax.annotate("", (7.75, 0.75), (7.0, 1.45), arrowprops=dict(color="#4a6d8c", **arrow))
ax.annotate("", (7.75, 0.6), (3.3, 1.18), arrowprops=dict(color="#0f7f58", connectionstyle="arc3,rad=0.25", **arrow))
ax.text(2.1, 1.75, "наблюдения", fontsize=8, ha="center", color="#52514e")
ax.text(4.7, 1.75, "сток − резерв", fontsize=8, ha="center", color="#52514e")
ax.text(7.5, 2.25, "суточная потребность", fontsize=8, ha="center", color="#52514e")
ax.text(7.4, 1.0, "избыток", fontsize=8, ha="center", color="#52514e")
ax.text(5.0, 0.35, "резерв = min(доля × сток, предел)", fontsize=8, ha="center", color="#0f7f58")
plt.show()

| Блок | Тип | Правило | На реальной реке |
|:------|:-----|:-----|:------------------|
| Приток | Source | передаёт измеренный суточный сток | Зарафшан на гидропосту Раватходжа |
| Гидроузел | Reach с `EFlowSplit` | оставляет в реке min(доля × сток, предел), остальное передаёт дальше | Раватходжинский гидроузел |
| Водозабор канала | Reach с `PriorityDistribution` | подаёт полям суточную потребность, остальное возвращает в реку | головная часть Самаркандских каналов |
| Поля | Sink | записывает поступление воды культурам | Даргом, Мирзапай, Аккарадья |
| Конец реки | Sink | записывает поступление воды для природы | Зарафшан ниже Раватходжи |

Между блоками нет накопления и потерь, поэтому каждый день *приток = вода в конце реки + вода полям*. Модель работает с суточным шагом, один календарный год за запуск; пять лет 2019–2023 оцениваются совместно.

## Река {#sec-river}

In [ ]:
# @title Суточные наблюдения Раватходжи за 2010–2023 в модели:...
#| label: river
# Суточные наблюдения Раватходжи за 2010–2023 в модели: столбцы date и q (м³/с)
inflow = load_inflow()
by_year = inflow.group_by(inflow["date"].dt.year().alias("year")).agg(pl.col("q").mean().round(1).alias("mean_m3s"), pl.col("q").min().alias("min_m3s"), pl.col("q").max().alias("max_m3s")).sort("year")
by_year

In [ ]:
# @title Рисунок: опорный ряд и потребность канала
#| label: fig-river
#| fig-cap: "Естественный опорный ряд (серый, все 14 лет) и потребность канала (зелёный). Пять оцениваемых лет выделены фоном."
#| echo: false
curve = np.array(demand_curve())
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(inflow["date"], inflow["q"], color="#878680", lw=0.7, label="Наблюдения Раватходжи")
ax.axvspan(np.datetime64("2019-01-01"), np.datetime64("2023-12-31"), color="#0f7f58", alpha=0.08, label="оцениваемые годы")
dates = inflow["date"].to_list()
ax.plot(dates, curve[[d.month - 1 for d in dates]], color="#007a00", lw=1.2, label="потребность канала (среднемесячная 2017–2022)")
ax.set_ylabel("m³/s"); ax.legend(loc="upper left", frameon=False); ax.set_xlim(dates[0], dates[-1])
plt.show()

Два факта определяют всё дальнейшее. Потребность канала мала относительно реки: в среднем `{python} f"{np.mean(curve):.1f}"` м³/с при среднем расходе `{python} f"{inflow['q'].mean():.0f}"` м³/с; с ноября по февраль потребности нет. Но вода требуется в неудобное для реки время: отбор начинается в марте, когда ещё продолжается зимняя межень, достигает пика в августе и в сентябре всё ещё составляет `{python} f"{curve[8]:.0f}"` м³/с, хотя летнее половодье уже прошло.

# Правило с двумя параметрами {#sec-rule}

`EFlowSplit` в TaqSim резервирует экологический сток до остальных распределений. У него два параметра:

- `eflow_fraction` — **резервируемая доля** суточного стока (от 0 до 1);
- `eflow_cap` — **верхний предел резерва**: резерв не превышает этот объём в сутки.

Суточный резерв равен *min(доля × сток, предел)*. Поэтому доля 100 % с пределом 60 м³/с означает: первые 60 м³/с каждого дня остаются в реке. Доля 50 % без предела означает: остаётся половина суточного стока. Затем правило `PriorityDistribution` у водозабора сначала снабжает поля в пределах месячной потребности, а остаток возвращает в реку.

In [ ]:
# @title Функция, соединяющая пять блоков для одного календарного года...
#| label: rule-source
# Функция, соединяющая пять блоков для одного календарного года при одном плане. Прочитайте сверху вниз:
# суточная ось времени, приток как IntervalVolume, два конечных узла и два участка с правилами.
print(inspect.getsource(build_system))

Несколько планов словами, как они называются в игре:

In [ ]:
# @title Правило с двумя параметрами · код
#| label: plan-names
for plan in (Plan(0, None), Plan(50, None), Plan(100, 60.0), Plan(50, 60.0), Plan(100, None)):
    print(f"доля {plan.share_pct:3d} %, предел {'нет' if plan.ceiling_m3s is None else f'{plan.ceiling_m3s:.0f} m³/s':10s} → {plan.name}")

# Соберите модель и рассчитайте один год {#sec-run}

Рассчитаем самый маловодный из оцениваемых лет, 2021, с планом «первые 60 м³/с всегда остаются». Запуск занимает несколько секунд: новый расчётный механизм TaqSim записывает каждое распределение, позволяя воспроизвести и проверить расчёт.

In [ ]:
# @title Сборка и расчёт одного года · код
#| label: one-run
plan = Plan(100, 60.0)
run = run_year(2021, plan)  # YearRun: даты, естественный сток, река и поля в м³/с, время расчёта
demand = m.demand_daily(run.dates)
crops_2021 = 100 * run.fields.sum() / demand.sum()
Markdown(
    m.engine_note() + f" Время расчёта {run.seconds:.1f} с для {len(run.dates)} суточных шагов. Невязка баланса {run.mass_balance_residual_m3():.0f} м³. "
    f"Поля получили {crops_2021:.1f} % потребности. Минимум ниже гидроузла {run.river.min():.1f} м³/с "
    f"(естественный {run.natural.min():.1f})."
)

In [ ]:
# @title Рисунок: 2021, первые 60 м³/с всегда остаются
#| label: fig-one-run
#| fig-cap: "2021 год при правиле «первые 60 м³/с всегда остаются». Зелёная полоса — вода, отобранная каналом."
#| echo: false
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(run.dates, run.natural, color="#878680", lw=1, ls="--", label="естественный")
ax.plot(run.dates, run.river, color="#4a6d8c", lw=1.4, label="река ниже гидроузла")
ax.fill_between(run.dates, run.river, run.natural, color="#007a00", alpha=0.3, label="отбор в канал")
ax.axhline(60, color="#0f7f58", lw=1, ls=":", label="гарантированный минимум 60 м³/с")
ax.set_ylabel("m³/s"); ax.legend(frameon=False, loc="upper left")
plt.show()

Те же два правила умещаются в четырёх строках numpy. Модель хранит этот параллельный расчёт для проверки каждого запуска TaqSim: результаты совпадают с точностью до дискретности расчётного механизма — 1 м³ в сутки.

In [ ]:
# @title Сборка и расчёт одного года · код
#| label: twin
print(inspect.getsource(twin_year))
river_twin, fields_twin = twin_year(run.natural, demand, plan)
print("наибольшая разница TaqSim − параллельный расчёт:", f"{np.abs(river_twin - run.river).max() * DAY:.2f} m³/day")

# Оцените состояние реки {#sec-score}

Оценка реки — индекс IARI из fishy. Для него нужны 33 показателя IHA естественного ряда по отдельным годам и те же 33 показателя изменённого ряда за последние пять лет. Для каждого показателя проверяется, попало ли пятилетнее значение в естественный межквартильный диапазон (между 25-м и 75-м процентилями). Если да, оценка равна 0; если нет — расстоянию за границей диапазона, делённому на его ширину. Индекс — среднее по 33 показателям. Границы ISPRA: ≤ 0.05 — близко к естественному, ≤ 0.15 — хорошее состояние, выше — изменённое.

Перед числами важны две оговорки. Метод требует не менее 20 опорных лет; ряд Раватходжи содержит 14 (2010–2023). Руководитель проекта решил использовать доступные данные, поэтому модель изменяет проверку fishy (см. `model._fishy`). Если fishy не может оценить один из показателей, общий индекс не возвращается; тогда игра показывает среднее доступных показателей, их количество и причины пропусков.

In [ ]:
# @title Оценка двух планов за пять лет 2019–2023. run_grid рассчитывает...
#| label: score-two-plans
# Оценка двух планов за пять лет 2019–2023. run_grid рассчитывает годы параллельно (около десяти секунд).
# Два примера со страницы игры: доля и минимум, выбранные при экспорте
examples = [Plan(p["share_pct"], p["ceiling_m3s"]) for p in DATA["plans"] if p["id"] in META["teaching_pair"]]
runs = m.run_grid(examples, workers=5)
results = {p.id: m.evaluate_plan(p, runs[p.id]) for p in examples}
for r in results.values():
    s = r.river
    print(f"{r.plan.name:32s} культуры {r.crops_pct:5.1f} %  оценка {s.score:.3f} ({'все 33' if s.total is not None else f'{s.scored} из 33'} показателей)  класс {s.classification}")
    for w in r.warnings:
        print("   ⚠", w)

Почти одинаковый объём для культур, но разное состояние реки. Какие показатели создают эту разницу?

In [ ]:
# @title Рисунок: 33 показателя двух планов
#| label: fig-indicators
#| fig-cap: "Оценки 33 показателей двух планов. Чем длиннее полоса, тем больше отклонение от естественного диапазона."
#| echo: false
a, b = results.values()
names = [k for k in a.river.indicators]
fig, ax = plt.subplots(figsize=(10, 8))
y = np.arange(len(names))
ax.barh(y - 0.2, [a.river.indicators[k] or 0 for k in names], height=0.4, color="#1f66c1", label=a.plan.name)
ax.barh(y + 0.2, [b.river.indicators[k] or 0 for k in names], height=0.4, color="#0f7f58", label=b.plan.name)
ax.set_yticks(y); ax.set_yticklabels([k.replace("_", " ") for k in names], fontsize=8); ax.invert_yaxis()
ax.axvline(0.05, color="#878680", lw=0.8, ls=":"); ax.axvline(0.15, color="#878680", lw=0.8, ls=":")
ax.legend(frameon=False); ax.set_xlabel("оценка показателя (0 = внутри естественного межквартильного диапазона)")
plt.show()

Правило доли отбирает больше всего воды в абсолютных величинах в дни высокого стока, что почти не влияет на индекс: эти дни остаются в естественном диапазоне. Но оно отбирает свою долю и в маловодные весенние дни, когда канал только начал работу: минимумы за 1, 3 и 7 дней и средние расходы весенних месяцев выходят за естественные границы. Гарантированный минимум в такие дни не допускает отбора. (Для плана A доступны лишь 32 из 33 показателей, поэтому его оценка — среднее по 32; ниже дано среднее плана B по тем же 32.)

In [ ]:
# @title Оценка реки · код
#| label: common-indicators
common = [k for k, v in a.river.indicators.items() if v is not None and b.river.indicators[k] is not None]
for r in (a, b):
    print(f"{r.plan.name:32s} среднее по {len(common)} общим доступным показателям: {np.mean([r.river.indicators[k] for k in common]):.3f}")

# Все планы одновременно {#sec-grid}

Страница игры содержит сетку всех сочетаний параметров: доли 0–100 % с шагом 10, пределы 10, 20, 40, 60, 80, 100, 150 и 200 м³/с или отсутствие предела. При доле 0 % любой предел даёт один и тот же план, поэтому он представлен один раз: `{python} len(DATA["plans"])` план, каждый рассчитан в TaqSim за 2019–2023 и оценён через fishy. Загрузка готовых результатов вместо повторного расчёта сокращает работу блокнота.

In [ ]:
# @title Все планы одновременно · код
#| label: grid-table
PLANS.filter(pl.col("unbeaten")).sort("crops_pct", descending=True)

In [ ]:
# @title Рисунок: все планы — культуры и оценка реки
#| label: fig-tradeoff
#| fig-cap: "Все планы: обеспеченность культур и оценка реки. Закрашенные точки — планы, которые ни один другой не превосходит. Правее лучше для культур, ниже лучше для реки."
#| echo: false
def family(row):
    if row["share_pct"] == 0 or (row["share_pct"] == 100 and row["ceiling_m3s"] is None):
        return "крайние планы"
    if row["ceiling_m3s"] is None:
        return "оставить долю"
    if row["share_pct"] == 100:
        return "гарантировать минимум"
    return "доля с пределом"
colours = {"крайние планы": "#0b0b0b", "оставить долю": "#1f66c1", "гарантировать минимум": "#0f7f58", "доля с пределом": "#9a9993"}
fig, ax = plt.subplots(figsize=(9, 5))
ax.axhspan(0, 0.05, color="#0f7f58", alpha=0.14); ax.axhspan(0.05, 0.15, color="#0f7f58", alpha=0.07)
for fam, colour in colours.items():
    rows = [r for r in DATA["plans"] if family(r) == fam]
    for r in rows:
        ax.scatter(r["crops_pct"], r["score"], s=46 if r["unbeaten"] else 26, facecolor=colour if r["unbeaten"] else "white", edgecolor=colour, lw=1.4, zorder=3 if r["unbeaten"] else 2)
    ax.scatter([], [], facecolor=colour, edgecolor=colour, label=fam)
pair = META["teaching_pair"]
for label, pid in zip("AB", pair):
    r = next(p for p in DATA["plans"] if p["id"] == pid)
    ax.annotate(label, (r["crops_pct"], r["score"]), xytext=(6, 6), textcoords="offset points", fontweight="bold")
ax.set_xlabel("обеспеченность культур, % потребности канала за 2019–2023"); ax.set_ylabel("оценка реки (IARI), 0 = естественный режим")
ax.legend(frameon=False, loc="upper left"); ax.set_xlim(-2, 102); ax.set_ylim(0, None)
plt.show()

# На что обратить внимание {#sec-notice}

In [ ]:
# @title На что обратить внимание · код
#| label: notice
#| echo: false
start = next(p for p in DATA["plans"] if p["id"] == META["starting_plan"])
A, B = (next(p for p in DATA["plans"] if p["id"] == pid) for pid in META["teaching_pair"])
unbeaten = [p for p in DATA["plans"] if p["unbeaten"]]
share_only = [p for p in DATA["plans"] if p["ceiling_m3s"] is None and 0 < p["share_pct"] < 100]
good_bound = META["iari_classes"]["buono"]
unchanged = max(p["share_pct"] for p in share_only if abs(p["score"] - start["score"]) < 5e-4)
last_altered = max(p["share_pct"] for p in share_only if p["score"] > good_bound)
first_good = min((p for p in share_only if p["score"] <= good_bound), key=lambda p: p["share_pct"])
scored = inflow.filter(pl.col("date").dt.year().is_in(list(m.SCORE_YEARS)))
canal_share = 100 * np.minimum(m.demand_daily(scored["date"].to_list()), scored["q"].to_numpy()).sum() / scored["q"].sum()
Markdown(f"""
1. **Канал забирает мало, но меняет реку.** При полном удовлетворении канала ("{start['name']}") культуры получают {start['crops_pct']:.0f} %, оценка реки {start['score']:.3f}: {start['classification'] or 'изменённое'}, среднее по {start['scored']} из 33 показателей. Канал забирает {canal_share:.0f} % воды пяти лет, но воздействует дважды: в апреле, когда ещё продолжается зимняя межень, и в сентябре, когда после спада половодья всё ещё требует {curve[8]:.0f} м³/с.
2. **Небольшая доля почти ничего не меняет.** Среди планов только с долей {sum(p['crops_pct'] > 99.5 for p in share_only)} из {len(share_only)} не меняют обеспеченность культур: канал редко требует больше остатка. Оценка не меняется до {unchanged} %, состояние остаётся изменённым до {last_altered} % и становится хорошим только при {first_good['share_pct']} %, когда культуры получают {first_good['crops_pct']:.1f} %.
3. **Та же вода, другая река.** Пример A ("{A['name']}") даёт культурам {A['crops_pct']:.1f} % при оценке {A['score']:.3f}; пример B ("{B['name']}") — {B['crops_pct']:.1f} % при {B['score']:.3f}. Минимум защищает маловодные дни, доля — нет.
4. **Недоминируемые планы.** Для {len(unbeaten)} из {len(DATA['plans'])} планов нет другого плана, превосходящего их; {sum(p['share_pct'] == 100 and p['ceiling_m3s'] is not None for p in unbeaten)} из них задают гарантированный минимум. Выбор между ними — решение о компромиссе между рекой и культурами, а не результат вычисления.
""")


# Ограничения {#sec-limits}

- **Один канал.** Реальный гидроузел Раватходжи также питает переброски в Джизак и Кашкадарью и другие каналы; здесь воду забирают только три Самаркандских канала. Поэтому реальная доля отбора больше, а компромисс острее.
- **Нет накопления, потерь и возвратного стока.** В канале нет фильтрации, с полей нет возврата, вода не переносится между днями. Игра изучает правило гидроузла, а не весь бассейн.
- **Фиксированная кривая потребности.** Канал каждый год запрашивает одинаковые месячные объёмы; маловодный год не меняет потребности культур.
- **Одно правило на все дни.** Реальные правила экологического стока меняются по сезонам и типам года; в исследованиях их разрабатывают сразу для многих целей (Suen & Eheart 2006; Chen & Olden 2017). Здесь оба параметра неизменны весь год.
- **Опора на современный режим.** Сток Зарафшана у Раватходжи регулируется выше по течению; 2010–2023 — современный ряд, а не нетронутая река. В нём 14 лет вместо требуемых 20, поэтому естественные квартили менее определённы, чем предполагает метод; пять оцениваемых лет входят и в опорный период.
- **Один индекс.** IARI измеряет изменение гидрологического режима, но ничего не говорит о рыбе, температуре и наносах.

# Упражнения {#sec-exercises}

1. Рассчитайте `Plan(100, 40.0)` за 2021 год через `run_year` и проверьте минимальный расход ниже гидроузла. Почему он не равен 40 м³/с? (См. естественный минимум в @sec-river.)
2. Добавьте третье семейство правил: долю, действующую только с марта по май. Само правило `EFlowSplit` этого не делает; предложите дополнительный блок и его правило.
3. Замените кривую потребности суммой шести каналов из файла потребностей (`m.SAMARKAND_CANALS` и три других столбца), пересчитайте два примера. Как меняется компромисс?
4. В таблице оценок найдите план с лучшей оценкой реки среди планов, обеспечивающих культуры более чем на 95 %. Это доля, минимум или доля с пределом?

# Источники {#sec-sources}

- Suen, J.-P. & Eheart, J. W. (2006). Reservoir management to balance ecosystem and human needs: incorporating the paradigm of the ecological flow regime. *Water Resources Research* 42, W03417.
- Chen, W. & Olden, J. D. (2017). Designing flows to resolve human and environmental water needs in a dam-regulated river. *Nature Communications* 8, 2158.
- Richter, B. D., Baumgartner, J. V., Powell, J. & Braun, D. P. (1996). A method for assessing hydrologic alteration within ecosystems. *Conservation Biology* 10, 1163–1174.
- ISPRA (2011). Implementazione della Direttiva 2000/60/CE: analisi e valutazione degli aspetti idromorfologici. Manuali e Linee Guida 131/2011. Индекс IARI и границы классов в реализации fishy.
- hydrosolutions. *fishy*: диагностика экологического стока для TaqSim. github.com/hydrosolutions/fishy, коммит 881660e.
- hydrosolutions. *TaqSim*. github.com/hydrosolutions/taqsim, коммит 396ad09.
- Данные: `data/ZRB_baseline/inflow/inflow_ravatkhoza_daily_2010_2023.csv` (суточный сток Раватходжи) и `data/ZRB_baseline/demand/demand_all_districts_2017-2022_monthly.csv` (месячные потребности каналов), оба файла в репозитории Zarafshan TaqSim.